<a href="https://colab.research.google.com/github/rakesh-mandal/ML/blob/main/XGBoost/XGBoost_Reg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import time
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import randint, uniform
from sklearn.datasets import load_diabetes
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV, KFold
)
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # disease progression score, 1 year after baseline


In [3]:
X.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [19]:
print(X.shape)
print(y.shape)

(442, 10)
(442,)


In [4]:
y

array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
        69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
        68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
        87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
       259.,  53., 190., 142.,  75., 142., 155., 225.,  59., 104., 182.,
       128.,  52.,  37., 170., 170.,  61., 144.,  52., 128.,  71., 163.,
       150.,  97., 160., 178.,  48., 270., 202., 111.,  85.,  42., 170.,
       200., 252., 113., 143.,  51.,  52., 210.,  65., 141.,  55., 134.,
        42., 111.,  98., 164.,  48.,  96.,  90., 162., 150., 279.,  92.,
        83., 128., 102., 302., 198.,  95.,  53., 134., 144., 232.,  81.,
       104.,  59., 246., 297., 258., 229., 275., 281., 179., 200., 200.,
       173., 180.,  84., 121., 161.,  99., 109., 115., 268., 274., 158.,
       107.,  83., 103., 272.,  85., 280., 336., 281., 118., 317., 235.,
        60., 174., 259., 178., 128.,  96., 126., 28

In [5]:
# 2. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}\n")

Train size: 353, Test size: 89



In [6]:
# 3. Train XGBoost regressor
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    eval_metric="rmse",
)

In [7]:
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False,
)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [8]:
# 4. Evaluate
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

In [9]:
print("=== Test set performance ===")
print(f"RMSE: {rmse:.2f}")
print(f"MAE:  {mae:.2f}")
print(f"R^2:  {r2:.3f}\n")

=== Test set performance ===
RMSE: 56.35
MAE:  45.84
R^2:  0.401



In [10]:
# 5. Feature importance
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("=== Feature importances ===")
print(importances)

=== Feature importances ===
bmi    0.222221
s5     0.194490
s4     0.111868
bp     0.094710
sex    0.072585
s6     0.070294
s3     0.064194
s2     0.059415
s1     0.057016
age    0.053207
dtype: float32


In [13]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
base_model = xgb.XGBRegressor(random_state=42, eval_metric="rmse")

In [14]:
param_grid = {
    "max_depth": [2, 3, 4],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 200, 300],
    "subsample": [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
}

In [15]:
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=0,
)


In [16]:
n_combos = int(np.prod([len(v) for v in param_grid.values()]))
print(f"GridSearchCV: testing {n_combos} combinations x {cv.get_n_splits()} folds "
      f"= {n_combos * cv.get_n_splits()} total fits")

GridSearchCV: testing 108 combinations x 5 folds = 540 total fits


In [17]:
t0 = time.time()
grid_search.fit(X_train, y_train)
grid_time = time.time() - t0

In [20]:
grid_search.best_params_

{'colsample_bytree': 0.7,
 'learning_rate': 0.05,
 'max_depth': 3,
 'n_estimators': 100,
 'subsample': 0.7}

In [21]:
grid_search.best_estimator_

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric='rmse', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [22]:
grid_best = grid_search.best_estimator_
grid_preds = grid_best.predict(X_test)
grid_rmse = np.sqrt(mean_squared_error(y_test, grid_preds))
grid_r2 = r2_score(y_test, grid_preds)

print(f"GridSearchCV done in {grid_time:.1f}s")
print(f"Best params: {grid_search.best_params_}")
print(f"Test RMSE: {grid_rmse:.2f}  |  Test R^2: {grid_r2:.3f}\n")

GridSearchCV done in 30.9s
Best params: {'colsample_bytree': 0.7, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.7}
Test RMSE: 51.91  |  Test R^2: 0.491



In [23]:
#============================================================
# RandomizedSearchCV — samples N_ITER random combos from
# DISTRIBUTIONS (can cover a much wider/finer space at fixed cost)
# ============================================================
param_dist = {
    "max_depth": randint(2, 8),
    "learning_rate": uniform(0.01, 0.29),      # 0.01 - 0.30
    "n_estimators": randint(100, 600),
    "subsample": uniform(0.5, 0.5),            # 0.5 - 1.0
    "colsample_bytree": uniform(0.5, 0.5),     # 0.5 - 1.0
    "reg_lambda": uniform(0.0, 5.0),
    "reg_alpha": uniform(0.0, 2.0),
    "min_child_weight": randint(1, 10),
}

In [24]:
N_ITER = 100  # number of random combinations to try (fixed cost, regardless of grid size)

In [25]:
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=N_ITER,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

In [26]:
print(f"RandomizedSearchCV: testing {N_ITER} random combinations "
      f"x {cv.get_n_splits()} folds = {N_ITER * cv.get_n_splits()} total fits")

RandomizedSearchCV: testing 100 random combinations x 5 folds = 500 total fits


In [27]:
t0 = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - t0

In [28]:
random_best = random_search.best_estimator_
random_preds = random_best.predict(X_test)
random_rmse = np.sqrt(mean_squared_error(y_test, random_preds))
random_r2 = r2_score(y_test, random_preds)

print(f"RandomizedSearchCV done in {random_time:.1f}s")
print(f"Best params: {random_search.best_params_}")
print(f"Test RMSE: {random_rmse:.2f}  |  Test R^2: {random_r2:.3f}\n")

RandomizedSearchCV done in 49.4s
Best params: {'colsample_bytree': np.float64(0.6886420815523113), 'learning_rate': np.float64(0.015820647355540646), 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 330, 'reg_alpha': np.float64(0.17840865742411238), 'reg_lambda': np.float64(2.6766778200776176), 'subsample': np.float64(0.6166082057749006)}
Test RMSE: 52.13  |  Test R^2: 0.487



In [29]:
# ============================================================
# Comparison
# ============================================================
print("=== GridSearchCV vs RandomizedSearchCV ===")
print(f"{'Method':<20}{'Time (s)':>10}{'RMSE':>10}{'R2':>10}")
print(f"{'GridSearchCV':<20}{grid_time:>10.1f}{grid_rmse:>10.2f}{grid_r2:>10.3f}")
print(f"{'RandomizedSearchCV':<20}{random_time:>10.1f}{random_rmse:>10.2f}{random_r2:>10.3f}")

=== GridSearchCV vs RandomizedSearchCV ===
Method                Time (s)      RMSE        R2
GridSearchCV              30.9     51.91     0.491
RandomizedSearchCV        49.4     52.13     0.487
